# 04 · Vetting features: turning clues into numbers

The pipeline finds **signals**, but many signals aren't planets. Most impostors are **eclipsing binaries**: two stars orbiting each other, eclipsing one another. Telling them apart needs evidence, and each clue spotted by eye in earlier notebooks becomes a number here, so a classifier (notebook 05) can use it.

| Clue | Feature | Planet | Impostor |
|---|---|---|---|
| U vs V shape | `shape_ratio` | ~0.8 to 1.0 (U) | ~0.3 to 0.6 (V) |
| Alternating eclipse depths | `odd_even_sigma` | low | high (a binary found at half its period) |
| A dip at phase 0.5 | `secondary_sigma`, `secondary_ratio` | none, or tiny (a hot planet's glow) | often clear (the companion passing behind) |
| Fraction of the orbit spent in transit | `duty_cycle` | ~0.01 to 0.1 | larger (binaries, continuous variables) |
| Correlated noise | `red_noise_beta`, `snr_red` | beta ≈ 1 | beta ≫ 1 (variable stars) |
| Inconsistent transits | `depth_chi2` | ≈ 1 | ≫ 1 |
| One event carrying the signal | `max_transit_frac` | small | close to 1 |
| Star's own variability | `variability_ppm` | varies | often high |
| Light's centre moving in transit | `centroid_sigma`, `centroid_source_offset_px` | low | high (a background binary) |

Three more features, computed in notebook 05 from the star's catalog properties: `planet_radius_re` (too big to be a planet?), `expected_duration_days` and `duration_ratio` (does the transit last as long as a planet around *this* star should?).

**One rule: no leakage.** Features come only from what the pipeline *measured*, never from the catalog's own period or depth for the signal. A model that saw the catalog's answers would look brilliant here and fail on new stars.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", message=".*oktopus.*")

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import lightkurve as lk
from src.pipeline import process_star
from src.features import compute_features, FEATURE_COLUMNS

---
## Test 1: Synthetic data, where the right answer is known

Before trusting the features on real stars, check them on two **fake** light curves built with known properties:

- **A planet:** a U-shaped, 0.5% transit every 2 days, every transit identical
- **A binary at half its period:** V-shaped dips, alternating between 0.6% and 0.3% deep

If the features don't separate these cleanly, they won't work on real data.

In [2]:
rng = np.random.default_rng(0)
t = np.arange(0, 27, 2 / 1440) # 27 days at 2-minute cadence
t = t[(t < 13) | (t > 14)] # a mid-sector data gap
P, T, t0 = 2.0, 0.12, 0.5 # period, duration, first transit (days)
phase = (t - t0 + P / 2) % P - P / 2
epoch = np.floor((t - t0 + P / 2) / P).astype(int)
x = np.abs(phase) / (T / 2) # 0 at mid-transit, 1 at the edges

# Planet: flat bottom with short sloped edges (a U)
u_dip = np.where(x < 0.9, 0.005, np.where(x < 1, 0.005 * (1 - x) / 0.1, 0))
planet = lk.LightCurve(time=t, flux=1 - u_dip + rng.normal(0, 300e-6, t.size))

# Binary at half its period: V-shaped dips, alternating deep and shallow
v_dip = np.clip(1 - x, 0, None) * np.where(epoch % 2 == 0, 0.006, 0.003)
binary = lk.LightCurve(time=t, flux=1 - v_dip + rng.normal(0, 300e-6, t.size))

synthetic = pd.DataFrame({
    "planet": compute_features(planet, planet, P, t0, T),
    "binary": compute_features(binary, binary, P, t0, T),
})
synthetic.loc[["shape_ratio", "odd_even_sigma", "depth_chi2", "max_transit_frac"]]

,planet,binary
shape_ratio,0.981475,0.309299
odd_even_sigma,1.881531,71.053190
depth_chi2,0.804522,413.641100
max_transit_frac,0.074235,0.121028


The features separate them exactly as intended:

- **`shape_ratio`:** about 1 for the U-shaped planet, about 0.3 for the V-shaped binary
- **`odd_even_sigma`:** about 2 for the planet (noise), about 70 for the binary (alternating depths)
- **`depth_chi2`:** about 1 for the planet (consistent transits), hundreds for the binary

(This test also caught a bug: the first version counted deep transits as stellar variability. The variability measurement now excludes in-transit points.)

---
## Test 2: Three real stars

| Star | What it is |
|---|---|
| **WASP-18** | A confirmed hot Jupiter |
| **TIC 67309890** | A false positive the pipeline found at exactly half its catalog period: a likely binary |
| **TIC 329277372** | A false positive with a 2.3%-deep signal, too deep for a typical planet |

In [3]:
targets = {
    "WASP-18": "planet",
    "TIC 67309890": "false positive (binary at half period)",
    "TIC 329277372": "false positive (2.3% deep)",
}

rows = []
for tgt, lab in targets.items():
    s = process_star(tgt, max_sectors=3)
    s["label"] = lab
    rows.append(s)

df = pd.DataFrame(rows).set_index("target")
df[["label", "period_days", "depth_ppm", "snr"] + FEATURE_COLUMNS].T

target,WASP-18,TIC 67309890,TIC 329277372
label,planet,false positive (binary at half period),false positive (2.3% deep)
period_days,0.941463,0.581201,0.516931
depth_ppm,10831.897747,729.343331,8407.885615
snr,681.838107,25.191673,437.851813
duty_cycle,0.089223,0.110117,0.386899
depth_odd_ppm,10901.914622,361.385585,3242.717263
depth_even_ppm,10844.103131,1183.744801,3567.896747
odd_even_sigma,1.471846,11.356255,3.149162
secondary_depth_ppm,386.896004,-65.630086,-5078.991056
secondary_sigma,19.477355,-1.843654,-99.072335


### Reading the table

**TIC 67309890, the binary at half its period, is caught by odd/even.** Odd transits average about **360 ppm**, even ones about **1,180 ppm**: a factor of three, giving `odd_even_sigma` ≈ **11**. Its `shape_ratio` of **0.57** also leans V-shaped. There's no secondary dip at phase 0.5, as expected: at half the true period, the secondary eclipse is folded onto the primary, which is exactly what the odd/even test detects.

**TIC 329277372 is not a transit at all.** Every feature points the same way:

- `duty_cycle` ≈ **0.39**: the "transit" fills 39% of the orbit
- `snr` **438 → `snr_red` 12**: the standard SNR was inflated about 37×, because the "signal" is continuous variation, not random noise
- `red_noise_beta` ≈ **5.4**: strongly correlated noise
- `variability_ppm` ≈ **16,600**: the star swings by ~1.7% outside the "transit"
- A **negative** secondary depth: the star is *brighter* halfway through the orbit

It's most likely a **contact binary**: two stars so close they touch, brightening and dimming smoothly all the way around the orbit.

**WASP-18 looks like a planet, with two instructive exceptions:**

- **`secondary_sigma` ≈ 19, with a ~390 ppm dip at phase 0.5.** WASP-18 b really has a secondary eclipse: it's hot enough that TESS sees its own glow vanish when it passes behind the star (published depth roughly 340 ppm). So a secondary eclipse alone doesn't mean a binary. What matters is its size *relative to the transit*: `secondary_ratio` is **3.6%** here, while two similar stars could give 50% or more.
- **`centroid_sigma` ≈ 4.6**, though the shift is tiny (0.001 pixels). With SNR this high, even a tiny shift becomes "significant." Dividing by the depth gives `centroid_source_offset_px` ≈ **0.1 pixels**: the dip comes from the target star itself.

### When a feature's assumptions break

TIC 329277372's `centroid_source_offset_px` is **94 pixels**, which is physically meaningless: the formula assumes a real transit, and there isn't one. But its raw centroid shift, **0.32 pixels at ~2,400σ**, moving in lockstep with the brightness, suggests the variation comes partly from a **neighbouring star** leaking into the aperture.

Extreme values like this are fine for tree-based models (random forests, gradient boosting), which only ask whether a value is above or below a threshold. Models sensitive to scale, like logistic regression, would need such features clipped or log-transformed first.

---
## Summary

The features behave as designed on both synthetic and real data, each catching a different type of impostor:

- **Binaries found at half their period** → odd/even depths
- **V-shaped eclipses** → shape ratio
- **Continuous variables** → duty cycle, correlated noise, variability
- **Background binaries** → centroid shift scaled by depth

No single feature catches everything, which is why notebook 05 combines them in a classifier.